In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [10]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [12]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [13]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [14]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [15]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [16]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [17]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

# Feature Selection

In [18]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"shape_MajorAxisLength",
"shape_Elongation",
"glszm_LargeAreaLowGrayLevelEmphasis_CT_c16",
"shape_Sphericity",
"shape_MinorAxisLength",
"shape_SurfaceVolumeRatio",
"LBP_201_PET",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2",
"shape_Flatness",
"glszm_GrayLevelNonUniformityNormalized_PET_c04"
]

In [19]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [20]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [23]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [24]:
X_new

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,42.073251,0.600926,7442.066260,0.761164,25.282894,0.251218,0.000062,0.000959,0.535140,0.140496
1,24.613845,0.841579,238.870676,0.697049,20.714498,0.489853,0.000349,0.002776,0.367109,0.180556
2,48.030294,0.772821,12957.157420,0.565792,37.118833,0.278467,0.000000,0.001179,0.597785,0.251029
3,25.589900,0.847727,233.632355,0.684364,21.693241,0.474018,0.000000,0.002748,0.405730,0.190083
4,34.684750,0.831483,121.921768,0.503142,28.839789,0.563135,0.000399,0.002309,0.442406,0.160494
...,...,...,...,...,...,...,...,...,...,...
134,33.069705,0.680294,1528.464999,0.742102,22.497115,0.322021,0.000000,0.001347,0.523608,0.285714
135,41.043692,0.758193,41182.958565,0.722918,31.119023,0.227705,0.000079,0.000850,0.735524,0.200000
136,36.618802,0.770113,3964.057173,0.652963,28.200620,0.298398,0.000000,0.001111,0.648063,0.173469
137,45.870392,0.628897,5243.112516,0.724255,28.847736,0.252893,0.000054,0.000935,0.492193,0.263889


In [25]:
X_new_std

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,-0.098422,-0.681983,-0.227575,1.075042,-0.404946,-0.558082,-0.315438,-0.496657,0.072232,-1.152048
1,-1.209119,1.006304,-0.275315,0.242767,-0.804734,1.585077,1.967842,2.054251,-1.302158,-0.669778
2,0.280542,0.523938,-0.191023,-1.461057,0.630836,-0.313359,-0.805067,-0.187799,0.584632,0.178635
3,-1.147026,1.049432,-0.275350,0.078112,-0.719083,1.442865,-0.805067,2.014276,-0.986261,-0.555083
4,-0.568449,0.935477,-0.276090,-2.274305,-0.093676,2.243217,2.359078,1.398595,-0.686277,-0.911297
...,...,...,...,...,...,...,...,...,...,...
134,-0.671191,-0.125182,-0.266768,0.827589,-0.648734,0.077795,-0.805067,0.048554,-0.022089,0.596207
135,-0.163918,0.421312,-0.003951,0.578577,0.105783,-0.769256,-0.180035,-0.649927,1.711257,-0.435690
136,-0.445412,0.504939,-0.250626,-0.329498,-0.149611,-0.134362,-0.805067,-0.282932,0.995873,-0.755087
137,0.143137,-0.485754,-0.242149,0.595927,-0.092981,-0.543043,-0.374777,-0.529934,-0.279046,0.333455


In [26]:
MAASTRO_new 

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,50.002093,0.765178,9404.025346,0.668072,38.260495,0.215184,0.000026,0.000768,0.610062,0.232422
1,41.753334,0.776540,16468.863361,0.669961,32.423122,0.276092,0.000167,0.001213,0.504616,0.339506
2,44.375483,0.697164,3829.811759,0.624081,30.936983,0.298887,0.000000,0.001176,0.478604,0.171875
3,46.115989,0.574636,2212.056593,0.577624,26.499895,0.361096,0.000080,0.001192,0.446059,0.166667
4,54.394967,0.633419,20185.616389,0.630933,34.454789,0.251519,0.000035,0.000766,0.480378,0.301020
...,...,...,...,...,...,...,...,...,...,...
94,34.218615,0.882411,18006.593217,0.671754,30.194871,0.307574,0.000078,0.001137,0.577884,0.202216
95,51.046869,0.535802,1303.136099,0.632189,27.351039,0.289922,0.000054,0.000962,0.455642,0.157025
96,50.417953,0.716610,8928.353080,0.645548,36.130031,0.228184,0.000028,0.000621,0.631485,0.460317
97,44.901412,0.665145,13835.684015,0.727488,29.865942,0.223872,0.000000,0.000947,0.628338,0.314879


In [27]:
MAASTRO_new_std

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,0.405980,0.470316,-0.214572,-0.133381,0.730745,-0.881698,-0.596780,-0.764236,0.685053,-0.045370
1,-0.118774,0.550025,-0.167748,-0.108854,0.219907,-0.334696,0.524128,-0.139994,-0.177437,1.243797
2,0.048037,-0.006831,-0.251516,-0.704414,0.089853,-0.129977,-0.805067,-0.191744,-0.390198,-0.774281
3,0.158761,-0.866418,-0.262238,-1.307469,-0.298444,0.428717,-0.170569,-0.168865,-0.656393,-0.836983
4,0.685437,-0.454030,-0.143115,-0.615466,0.397702,-0.555379,-0.525146,-0.767510,-0.375685,0.780474
...,...,...,...,...,...,...,...,...,...,...
94,-0.598102,1.292755,-0.157556,-0.085581,0.024909,-0.051953,-0.187979,-0.246354,0.421853,-0.409011
95,0.472444,-1.138850,-0.268262,-0.599161,-0.223959,-0.210490,-0.373913,-0.492861,-0.578011,-0.953060
96,0.432435,0.129595,-0.217724,-0.425762,0.544305,-0.764952,-0.581052,-0.971544,0.860275,2.698219
97,0.081495,-0.231458,-0.185200,0.637891,-0.003876,-0.803671,-0.805067,-0.513058,0.834539,0.947314


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [28]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 19:07:05,305] A new study created in memory with name: no-name-b93d70ce-74b9-4ab5-8879-7427b2329300
python(43293) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7124463519313304
[I 2024-04-14 19:07:20,012] Trial 0 finished with value: 0.7028948264777959 and parameters: {}. Best is trial 0 with value: 0.7028948264777959.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7028948264777959], datetime_start=datetime.datetime(2024, 4, 14, 19, 7, 5, 476177), datetime_complete=datetime.datetime(2024, 4, 14, 19, 7, 20, 11421), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7028948264777959


[I 2024-04-14 19:07:20,119] A new study created in memory with name: no-name-d0b8e984-9d09-470d-8fd9-f8da9126effa


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.26188816761455547
Fold 2 IBS: 0.17457592134628122
Fold 3 IBS: 0.15644372418899136
Fold 4 IBS: 0.21056251915501756
Fold 5 IBS: 0.19139883133650912
[I 2024-04-14 19:07:20,941] Trial 0 finished with value: 0.19897383272827093 and parameters: {}. Best is trial 0 with value: 0.19897383272827093.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19897383272827093], datetime_start=datetime.datetime(2024, 4, 14, 19, 7, 20, 204827), datetime_complete=datetime.datetime(2024, 4, 14, 19, 7, 20, 940704), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19897383272827093


In [29]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [30]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.703
train_ibs:  0.199


#### Test

In [31]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [32]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.528
IBS score: 0.309


In [33]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [34]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:07:21,603] A new study created in memory with name: no-name-f5cff67d-c892-4577-bda7-16cbf96bb876


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5916334661354582
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7167300380228137


[I 2024-04-14 19:07:22,053] A new study created in memory with name: no-name-59c35267-2bad-465d-90d9-1bc58b6d2b5f


Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 19:07:22,041] Trial 0 finished with value: 0.6482287832820909 and parameters: {}. Best is trial 0 with value: 0.6482287832820909.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6482287832820909], datetime_start=datetime.datetime(2024, 4, 14, 19, 7, 21, 682902), datetime_complete=datetime.datetime(2024, 4, 14, 19, 7, 22, 41359), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6482287832820909


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709428728842
Fold 2 IBS: 0.2320398711590297
Fold 3 IBS: 0.22898186727039876
Fold 4 IBS: 0.24197476011764557
Fold 5 IBS: 0.22939558449397743
[I 2024-04-14 19:07:22,771] Trial 0 finished with value: 0.23592783546566798 and parameters: {}. Best is trial 0 with value: 0.23592783546566798.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783546566798], datetime_start=datetime.datetime(2024, 4, 14, 19, 7, 22, 129299), datetime_complete=datetime.datetime(2024, 4, 14, 19, 7, 22, 770562), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783546566798


In [36]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.648
train_ibs:  0.236


#### Test

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.537


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:07:23,680] A new study created in memory with name: no-name-097de088-6d7c-4392-995e-8cf51ad216ce


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8


[I 2024-04-14 19:07:25,167] A new study created in memory with name: no-name-cf8791d5-9dfb-40d0-a9db-a05938bb3333


Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:25,118] Trial 0 finished with value: 0.7011706560246108 and parameters: {}. Best is trial 0 with value: 0.7011706560246108.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7011706560246108], datetime_start=datetime.datetime(2024, 4, 14, 19, 7, 23, 739624), datetime_complete=datetime.datetime(2024, 4, 14, 19, 7, 25, 117782), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7011706560246108


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2618018981937393
Fold 2 IBS: 0.1754076638033244
Fold 3 IBS: 0.1575583206585483
Fold 4 IBS: 0.21083436582901882
Fold 5 IBS: 0.18978087462357812
[I 2024-04-14 19:07:26,597] Trial 0 finished with value: 0.19907662462164177 and parameters: {}. Best is trial 0 with value: 0.19907662462164177.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19907662462164177], datetime_start=datetime.datetime(2024, 4, 14, 19, 7, 25, 209960), datetime_complete=datetime.datetime(2024, 4, 14, 19, 7, 26, 595979), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19907662462164177


In [42]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.701
train_ibs:  0.199


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.528


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.308


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:07:27,720] A new study created in memory with name: no-name-eec4122a-7a1f-465d-9b77-ddb289b1d5ae


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:29,454] Trial 0 finished with value: 0.7020217198543981 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7020217198543981.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:30,613] Trial 1 finished with value: 0.7028727836841854 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7028727836841854.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:31,845] Trial 2 finished with value: 0.7036695964331894 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:50,404] Trial 24 finished with value: 0.7045206602629766 and parameters: {'l1_ratio': 0.07543723109624823}. Best is trial 10 with value: 0.7045206602629766.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:51,401] Trial 25 finished with value: 0.7036695964331894 and parameters: {'l1_ratio': 0.19147826708277826}. Best is trial 10 with value: 0.7045206602629766.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:07:52,711] Trial 26 finished with value: 0.7028727836841854 and parameters: {'l1_ratio': 0.3310420285446

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:08:14,297] Trial 48 finished with value: 0.7020217198543981 and parameters: {'l1_ratio': 0.7991933234425521}. Best is trial 10 with value: 0.7045206602629766.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:08:15,043] Trial 49 finished with value: 0.7028727836841854 and parameters: {'l1_ratio': 0.2943478497434463}. Best is trial 10 with value: 0.7045206602629766.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 19:08:15,770] Trial 50 finished with value: 0.7045206602629766 and parameters: {'l1_ratio': 0.169288242083098

...

Fold 1 IBS: 0.2615054568432813
Fold 2 IBS: 0.17546124558500803
Fold 3 IBS: 0.15745953203134078
Fold 4 IBS: 0.21081180496529733
Fold 5 IBS: 0.18981091835099756
[I 2024-04-14 19:08:51,456] Trial 0 finished with value: 0.199009791555185 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.199009791555185.
Fold 1 IBS: 0.26102096459211444
Fold 2 IBS: 0.17546310911346108
Fold 3 IBS: 0.15727383805312153
Fold 4 IBS: 0.21071976990897665
Fold 5 IBS: 0.18981867332992686
[I 2024-04-14 19:08:52,398] Trial 1 finished with value: 0.1988592709995201 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.1988592709995201.
Fold 1 IBS: 0.26095653589842793
Fold 2 IBS: 0.175541884629451
Fold 3 IBS: 0.15722363282072982
Fold 4 IBS: 0.21070881524532029
Fold 5 IBS: 0.18972106375483969
[I 2024-04-14 19:08:53,140] Trial 2 finished with value: 0.19883038646975373 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.19883038646975373.

Fold 1 IBS: 0.2606821812817422
Fold 2 IBS: 0.22738698140772204
Fold 3 IBS: 0.22888824044815342
Fold 4 IBS: 0.23799375816123436
Fold 5 IBS: 0.2263768358729022
[I 2024-04-14 19:09:05,362] Trial 25 finished with value: 0.23626559943435083 and parameters: {'l1_ratio': 0.03981317955666948}. Best is trial 22 with value: 0.1986950643681336.
Fold 1 IBS: 0.2609154703081647
Fold 2 IBS: 0.17545827162195377
Fold 3 IBS: 0.1571670244831515
Fold 4 IBS: 0.21068267014118114
Fold 5 IBS: 0.18976999274146958
[I 2024-04-14 19:09:06,028] Trial 26 finished with value: 0.19879868585918412 and parameters: {'l1_ratio': 0.18783928730207888}. Best is trial 22 with value: 0.1986950643681336.
Fold 1 IBS: 0.26079022916853595
Fold 2 IBS: 0.17520799790036354
Fold 3 IBS: 0.15704865024719103
Fold 4 IBS: 0.21063208131503916
Fold 5 IBS: 0.18970289615968972
[I 2024-04-14 19:09:06,663] Trial 27 finished with value: 0.19867637095816387 and parameters: {'l1_ratio': 0.07505393448412474}. Best is trial 27 with value: 0.19867637

Fold 5 IBS: 0.1898459598326078
[I 2024-04-14 19:09:22,435] Trial 49 finished with value: 0.19893060573157434 and parameters: {'l1_ratio': 0.4477016529538363}. Best is trial 27 with value: 0.19867637095816387.
Fold 1 IBS: 0.2457151388464811
Fold 2 IBS: 0.22873767739586323
Fold 3 IBS: 0.2288755391653703
Fold 4 IBS: 0.23915447826885858
Fold 5 IBS: 0.22726120269929223
[I 2024-04-14 19:09:22,655] Trial 50 finished with value: 0.23394880727517312 and parameters: {'l1_ratio': 0.02711267007130712}. Best is trial 27 with value: 0.19867637095816387.
Fold 1 IBS: 0.26084493493469907
Fold 2 IBS: 0.17528004420872453
Fold 3 IBS: 0.15703511863648179
Fold 4 IBS: 0.21063649084213493
Fold 5 IBS: 0.1897265008034076
[I 2024-04-14 19:09:23,422] Trial 51 finished with value: 0.19870461788508959 and parameters: {'l1_ratio': 0.08640058638065554}. Best is trial 27 with value: 0.19867637095816387.
Fold 1 IBS: 0.2608817355366732
Fold 2 IBS: 0.17541335158964097
Fold 3 IBS: 0.15714376879533745
Fold 4 IBS: 0.2106718

Fold 1 IBS: 0.2608575610544787
Fold 2 IBS: 0.17539348844468203
Fold 3 IBS: 0.15719466962952938
Fold 4 IBS: 0.2106838446583365
Fold 5 IBS: 0.1897136590361089
[I 2024-04-14 19:09:38,070] Trial 74 finished with value: 0.19876864456462712 and parameters: {'l1_ratio': 0.17727321357700984}. Best is trial 27 with value: 0.19867637095816387.
Fold 1 IBS: 0.2607981994290502
Fold 2 IBS: 0.1752317549019866
Fold 3 IBS: 0.15706047451460672
Fold 4 IBS: 0.21062862657188466
Fold 5 IBS: 0.18967484364348253
[I 2024-04-14 19:09:38,651] Trial 75 finished with value: 0.19867877981220214 and parameters: {'l1_ratio': 0.08245157736410111}. Best is trial 27 with value: 0.19867637095816387.
Fold 1 IBS: 0.2608709289390057
Fold 2 IBS: 0.17529454472507108
Fold 3 IBS: 0.1571042565950744
Fold 4 IBS: 0.21065490271630477
Fold 5 IBS: 0.18979072613153178
[I 2024-04-14 19:09:39,204] Trial 76 finished with value: 0.19874307182139755 and parameters: {'l1_ratio': 0.12378281768481025}. Best is trial 27 with value: 0.198676370

Fold 1 IBS: 0.26091206231297664
Fold 2 IBS: 0.17532142366602946
Fold 3 IBS: 0.15710800910675635
Fold 4 IBS: 0.21066952588141638
Fold 5 IBS: 0.18970528727277605
[I 2024-04-14 19:09:52,528] Trial 99 finished with value: 0.198743261647991 and parameters: {'l1_ratio': 0.1400329988695925}. Best is trial 89 with value: 0.19866962654598722.


* Best trial for IBS: 
 FrozenTrial(number=89, state=TrialState.COMPLETE, values=[0.19866962654598722], datetime_start=datetime.datetime(2024, 4, 14, 19, 9, 44, 790063), datetime_complete=datetime.datetime(2024, 4, 14, 19, 9, 45, 967885), params={'l1_ratio': 0.07349840790755927}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=89, value=None)


* Best Score for IBS: 
 0.19866962654598722


In [48]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.705
train_ibs:  0.199


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.08382735104105443)

test_cindex : 0.527


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07349840790755927)

test_ibs:  0.308


In [52]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 19:09:53,190] A new study created in memory with name: no-name-bf3e278e-07d7-45d8-a879-f5d1d6a0d99c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6573705179282868
Fold 2 C-index: 0.8023255813953488
Fold 3 C-index: 0.6297872340425532
Fold 4 C-index: 0.7414448669201521
Fold 5 C-index: 0.6931330472103004
[I 2024-04-14 19:10:02,463] Trial 0 finished with value: 0.7048122494993283 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7048122494993283.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7674418604651163
Fold 3 C-index: 0.6553191489361702
Fold 4 C-index: 0.7604562737642585
Fold 5 C-index: 0.7339055793991416
[I 2024-04-14 19:10:08,215] Trial 1 finished with value: 0.7045401103615428 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 3 C-index: 0.8170212765957446
Fold 4 C-index: 0.8479087452471483
Fold 5 C-index: 0.8197424892703863
[I 2024-04-14 19:11:09,979] Trial 15 finished with value: 0.7770080066545406 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 7, 'n_estimators': 170, 'oob_score': True, 'max_samples': 0.834513235692611, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1695220400209786, 'warm_start': True}. Best is trial 14 with value: 0.7773044783671873.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.8365019011406845
Fold 5 C-index: 0.8197424892703863
[I 2024-04-14 19:11:11,125] Trial 16 finished with value: 0.7738213229226775 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.804668454581114, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1700482219243455

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.8449612403100775
Fold 3 C-index: 0.8936170212765957
Fold 4 C-index: 0.8821292775665399
Fold 5 C-index: 0.871244635193133
[I 2024-04-14 19:11:40,218] Trial 30 finished with value: 0.8187091599688708 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 309, 'oob_score': True, 'max_samples': 0.9090216144690085, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.07553359551188188, 'warm_start': True}. Best is trial 30 with value: 0.8187091599688708.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.8294573643410853
Fold 3 C-index: 0.8936170212765957
Fold 4 C-index: 0.8821292775665399
Fold 5 C-index: 0.8583690987124464
[I 2024-04-14 19:11:43,010] Trial 31 finished with value: 0.8090492137339151 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 4, 'n_estimators': 307, 'oob_score': True, 'max_samples': 0.9009241102826697, '

...

Fold 1 IBS: 0.22871119834201115
Fold 2 IBS: 0.18629984266823443
Fold 3 IBS: 0.2136333562391605
Fold 4 IBS: 0.20508306074253652
Fold 5 IBS: 0.20460862507387245
[I 2024-04-14 19:18:44,946] Trial 32 finished with value: 0.20766721661316295 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 7, 'min_samples_leaf': 15, 'max_depth': 3, 'n_estimators': 310, 'oob_score': False, 'max_samples': 0.8321993039875358, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16244513867075333}. Best is trial 1 with value: 0.20613771601435493.
Fold 1 IBS: 0.2450153519251981
Fold 2 IBS: 0.19177067560329883
Fold 3 IBS: 0.2050380118625506
Fold 4 IBS: 0.19783492315190962
Fold 5 IBS: 0.20502773397972165
[I 2024-04-14 19:18:50,531] Trial 33 finished with value: 0.20893733930453579 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 7, 'max_depth': 4, 'n_estimators': 224, 'oob_score': False, 'max_samples': 0.9881033887741655, 'max_features': 'log2', 'min_weight_fraction_lea

Fold 1 IBS: 0.2464013758053453
Fold 2 IBS: 0.23207408714903174
Fold 3 IBS: 0.22939892327646114
Fold 4 IBS: 0.2415689721009919
Fold 5 IBS: 0.23043319376810475
[I 2024-04-14 19:19:38,369] Trial 48 finished with value: 0.23597531041998696 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 108, 'oob_score': True, 'max_samples': 0.6344615640154665, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.44962767788541247}. Best is trial 1 with value: 0.20613771601435493.
Fold 1 IBS: 0.24633584274807316
Fold 2 IBS: 0.23218539826865298
Fold 3 IBS: 0.22972627891694997
Fold 4 IBS: 0.2411567932158465
Fold 5 IBS: 0.2301175121753443
[I 2024-04-14 19:19:41,030] Trial 49 finished with value: 0.2359043650649734 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 13, 'min_samples_leaf': 20, 'max_depth': 2, 'n_estimators': 168, 'oob_score': False, 'max_samples': 0.2700643631363231, 'max_features': 'auto', 'min_weight_fraction_leaf'

Fold 1 IBS: 0.2263102371899488
Fold 2 IBS: 0.1935254758393429
Fold 3 IBS: 0.20737484396299696
Fold 4 IBS: 0.2086187676158567
Fold 5 IBS: 0.20696146438052143
[I 2024-04-14 19:20:27,835] Trial 64 finished with value: 0.2085581577977334 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 6, 'n_estimators': 252, 'oob_score': False, 'max_samples': 0.5041409232103772, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1346031511543957}. Best is trial 52 with value: 0.2056012262727213.
Fold 1 IBS: 0.23692845343363148
Fold 2 IBS: 0.19041679735965694
Fold 3 IBS: 0.21008157321222845
Fold 4 IBS: 0.1987406581683996
Fold 5 IBS: 0.20283869466958301
[I 2024-04-14 19:20:34,292] Trial 65 finished with value: 0.2078012353686999 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 3, 'n_estimators': 340, 'oob_score': False, 'max_samples': 0.689407190096443, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.2141097932587844
Fold 2 IBS: 0.20953167979731777
Fold 3 IBS: 0.25602778622576694
Fold 4 IBS: 0.2273921618025962
Fold 5 IBS: 0.2321408254663515
[I 2024-04-14 19:21:42,827] Trial 80 finished with value: 0.2278404493101634 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 7, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 2, 'oob_score': False, 'max_samples': 0.7067242004030547, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.3136324910319987}. Best is trial 52 with value: 0.2056012262727213.
Fold 1 IBS: 0.2300650150874554
Fold 2 IBS: 0.18010561006896428
Fold 3 IBS: 0.21441487116790292
Fold 4 IBS: 0.19757686931100543
Fold 5 IBS: 0.21275664692284824
[I 2024-04-14 19:21:46,629] Trial 81 finished with value: 0.20698380251163523 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 4, 'n_estimators': 70, 'oob_score': False, 'max_samples': 0.9023474857295949, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1

Fold 1 IBS: 0.22385859901871702
Fold 2 IBS: 0.18749154864023293
Fold 3 IBS: 0.215405777089575
Fold 4 IBS: 0.19883810395439896
Fold 5 IBS: 0.21146953699788537
[I 2024-04-14 19:22:22,820] Trial 96 finished with value: 0.20741271314016183 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 4, 'n_estimators': 58, 'oob_score': False, 'max_samples': 0.8577528441109724, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18660062214819917}. Best is trial 52 with value: 0.2056012262727213.
Fold 1 IBS: 0.2206463816639038
Fold 2 IBS: 0.18415089303949564
Fold 3 IBS: 0.2084960116600641
Fold 4 IBS: 0.20343560492304824
Fold 5 IBS: 0.20851133716775203
[I 2024-04-14 19:22:26,091] Trial 97 finished with value: 0.20504804569085272 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 7, 'max_depth': 3, 'n_estimators': 95, 'oob_score': True, 'max_samples': 0.8475291147257535, 'max_features': 'log2', 'min_weight_fraction_leaf': 

In [54]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.845
train_ibs:  0.205


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=17, max_features='log2', max_leaf_nodes=14,
                     max_samples=0.8948299547976223, min_samples_leaf=2,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.03150292204046534,
                     n_estimators=213, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.521


RandomSurvivalForest(max_depth=3, max_features='log2', max_leaf_nodes=10,
                     max_samples=0.8475291147257535, min_samples_leaf=7,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.20904972733074298,
                     n_estimators=95, oob_score=True, random_state=123)

test_ibs:  0.248


In [58]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [60]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:22:36,348] A new study created in memory with name: no-name-0d44bbd0-d67b-47ec-b543-31eb5cbe21fb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.8197424892703863
[I 2024-04-14 19:22:38,971] Trial 0 finished with value: 0.7503380732219547 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7503380732219547.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:22:44,347] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Best is tria

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.7725321888412017
[I 2024-04-14 19:23:38,271] Trial 16 finished with value: 0.7223376647099076 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.7665885346648663.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-14 19:23:40,308] Trial 17 finished with value: 0.7176426303552769 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.8022813688212928
Fold 5 C-index: 0.8154506437768241
[I 2024-04-14 19:24:14,936] Trial 31 finished with value: 0.7598850722510556 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 458, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9135692082875294, 'min_weight_fraction_leaf': 0.03048028383469985}. Best is trial 24 with value: 0.7703255353389682.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.8111587982832618
[I 2024-04-14 19:24:17,625] Trial 32 finished with value: 0.7497695648179512 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 3, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_featur

...

Fold 1 IBS: 0.23852023407558698
Fold 2 IBS: 0.22085127198360574
Fold 3 IBS: 0.22246208066369713
Fold 4 IBS: 0.22864256467493396
Fold 5 IBS: 0.21968838126903262
[I 2024-04-14 19:32:50,666] Trial 30 finished with value: 0.22603290653337127 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.21246282745647482.
Fold 1 IBS: 0.23299946909800737
Fold 2 IBS: 0.20723625804859605
Fold 3 IBS: 0.21221943785442013
Fold 4 IBS: 0.2177017964639846
Fold 5 IBS: 0.20854416781687213
[I 2024-04-14 19:32:57,065] Trial 31 finished with value: 0.21574022585637603 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 408, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 

Fold 1 IBS: 0.23535635522895523
Fold 2 IBS: 0.19710426247292917
Fold 3 IBS: 0.20557962919622366
Fold 4 IBS: 0.21419277453695665
Fold 5 IBS: 0.20885872177069337
[I 2024-04-14 19:34:08,849] Trial 45 finished with value: 0.2122183486411516 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 226, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.4468362954201356, 'min_weight_fraction_leaf': 0.022410036988178828}. Best is trial 35 with value: 0.21132948906228716.
Fold 1 IBS: 0.23697226819401304
Fold 2 IBS: 0.21729894679225342
Fold 3 IBS: 0.21924579151831378
Fold 4 IBS: 0.22736263277866023
Fold 5 IBS: 0.21648723912795567
[I 2024-04-14 19:34:14,734] Trial 46 finished with value: 0.2234733756822392 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 246, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.24665906141024754
Fold 2 IBS: 0.23238673943832674
Fold 3 IBS: 0.22958317859156768
Fold 4 IBS: 0.24137956784118675
Fold 5 IBS: 0.23028136399594556
[I 2024-04-14 19:35:33,038] Trial 60 finished with value: 0.23605798225545485 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 323, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.43142720229784703, 'min_weight_fraction_leaf': 0.48740241401755297}. Best is trial 56 with value: 0.20752977252523447.
Fold 1 IBS: 0.2321959013757636
Fold 2 IBS: 0.2007164455461997
Fold 3 IBS: 0.20770062793391472
Fold 4 IBS: 0.20602392627423452
Fold 5 IBS: 0.19819621614422855
[I 2024-04-14 19:35:36,340] Trial 61 finished with value: 0.20896662345486822 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 193, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.3

Fold 1 IBS: 0.2355921510352545
Fold 2 IBS: 0.18988955633106214
Fold 3 IBS: 0.2175269995788695
Fold 4 IBS: 0.1993059384582704
Fold 5 IBS: 0.19950554004549564
[I 2024-04-14 19:36:11,861] Trial 75 finished with value: 0.20836403708979043 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 7, 'n_estimators': 62, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.6761462774417768, 'min_weight_fraction_leaf': 0.07610410535003291}. Best is trial 72 with value: 0.2074917391497911.
Fold 1 IBS: 0.2331646523531711
Fold 2 IBS: 0.19585793938278617
Fold 3 IBS: 0.21617307743244904
Fold 4 IBS: 0.20830690666506493
Fold 5 IBS: 0.2045693433216475
[I 2024-04-14 19:36:14,122] Trial 76 finished with value: 0.21161438383102374 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 52, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.76838967

Fold 1 IBS: 0.2310593691941199
Fold 2 IBS: 0.19290862866485556
Fold 3 IBS: 0.2259889263099397
Fold 4 IBS: 0.20574406592712768
Fold 5 IBS: 0.2036881105655048
[I 2024-04-14 19:36:50,815] Trial 90 finished with value: 0.21187782013230955 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 33, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.6403451257560147, 'min_weight_fraction_leaf': 0.1070974205302174}. Best is trial 72 with value: 0.2074917391497911.
Fold 1 IBS: 0.2401929507969826
Fold 2 IBS: 0.19402821024539957
Fold 3 IBS: 0.20959009756624364
Fold 4 IBS: 0.19925984870201024
Fold 5 IBS: 0.20019981509015003
[I 2024-04-14 19:36:55,034] Trial 91 finished with value: 0.2086541844801572 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 131, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.69469242

In [61]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [62]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.837
train_ibs:  0.207


#### Test

In [63]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [64]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=18, max_features='auto', max_leaf_nodes=17,
                   max_samples=0.678838159065027, min_samples_leaf=1,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.011359003866057657,
                   n_estimators=362, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.509


ExtraSurvivalTrees(max_depth=7, max_features=None, max_leaf_nodes=15,
                   max_samples=0.7134910869727897, min_samples_split=2,
                   min_weight_fraction_leaf=0.0694822375534185, n_estimators=57,
                   random_state=123)

IBS: 0.259


In [65]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [66]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 19:37:22,920] A new study created in memory with name: no-name-ec2242a4-bf3b-43fd-a0d8-a76741461787


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:38:06,534] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:38:35,931] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:48:36,931] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6970416816836329.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:49:42,806] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:02:15,712] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.6970416816836329.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:02:48,885] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:28:58,968] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:29:49,957] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8324789538052517

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:41:29,068] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.010488456761899951, 'dropout_rate': 0.32975112844548055, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.7134499429690735, 'min_weight_fraction_leaf': 0.19311178079767077, 'max_features': 'log2', 'min_impurity_decrease': 1.142816958470248e-06, 'validation_fraction': 0.916501041728546, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 5}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:42:34,055] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9237031040797604

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:51:50,386] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9372636724132823, 'learning_rate': 0.04128077713454106, 'dropout_rate': 0.2214138929394412, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 0.5872217111247551, 'min_weight_fraction_leaf': 0.2869095730072229, 'max_features': 1, 'min_impurity_decrease': 1.637559261311236e-05, 'validation_fraction': 0.8634632458479912, 'min_samples_split': 19, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 14}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:53:15,391] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7443996478445325, 'lear

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 22:03:54,540] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5839343488674055, 'learning_rate': 0.005616464421601462, 'dropout_rate': 0.35953503133089737, 'n_estimators': 446, 'criterion': 'squared_error', 'ccp_alpha': 0.3179664432907911, 'min_weight_fraction_leaf': 0.28749157637391265, 'max_features': 'auto', 'min_impurity_decrease': 3.335913971167387e-07, 'validation_fraction': 0.9542099213957375, 'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 12, 'max_depth': 1}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24665149166978298
Fold 2 IBS: 0.23100341911248753
Fold 3 IBS: 0.22874931175992053
Fold 4 IBS: 0.241449076384833
Fold 5 IBS: 0.22889139804833578
[I 2024-04-14 22:05:26,646] Trial 89 finished with value: 0.23534893939507198 and parameters: {'subsample': 0.9747597567605316,

Fold 2 IBS: 0.23172776185253358
Fold 3 IBS: 0.22878554072857857
Fold 4 IBS: 0.2416676429021851
Fold 5 IBS: 0.2290838777915168
[I 2024-04-14 22:14:54,526] Trial 99 finished with value: 0.23565647433985792 and parameters: {'subsample': 0.24286963781937687, 'learning_rate': 0.008804936390818944, 'dropout_rate': 0.14353784252843382, 'n_estimators': 498, 'criterion': 'squared_error', 'ccp_alpha': 0.004530258324353442, 'min_weight_fraction_leaf': 0.29607175965989646, 'max_features': 0.1, 'min_impurity_decrease': 1.9888370412270107e-06, 'validation_fraction': 0.9539243209909428, 'min_samples_split': 19, 'max_leaf_nodes': 20, 'min_samples_leaf': 13, 'max_depth': 2}. Best is trial 53 with value: 0.2338206539157001.


* Best trial for IBS: 
 FrozenTrial(number=53, state=TrialState.COMPLETE, values=[0.2338206539157001], datetime_start=datetime.datetime(2024, 4, 14, 21, 26, 3, 299914), datetime_complete=datetime.datetime(2024, 4, 14, 21, 27, 6, 96288), params={'subsample': 0.9101431135071837, 'lea

In [67]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [68]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.711
train_ibs:  0.234


#### Test

In [69]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [70]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.02313127962403605,
                                 criterion='squared_error',
                                 dropout_rate=0.7413014948736192,
                                 learning_rate=0.03158550801216725,
                                 max_features='sqrt', max_leaf_nodes=20,
                                 min_impurity_decrease=1.9076132907505136e-05,
                                 min_samples_leaf=11, min_samples_split=18,
                                 min_weight_fraction_leaf=0.25025224628887144,
                                 n_estimators=449, random_state=123,
                                 subsample=0.5109023203910539,
                                 validation_fraction=0.8776959526028103)

C-index score: 0.517


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009625013743012712,
                                 criterion='squared_error',
                                 dropout_rate=0.1897783294507234,
                                 learning_rate=0.015420772490455037,
                                 max_features='auto', max_leaf_nodes=14,
                                 min_impurity_decrease=5.869825897765074e-07,
                                 min_samples_leaf=13, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29880213170319914,
                                 n_estimators=430, random_state=123,
                                 subsample=0.9101431135071837,
                                 validation_fraction=0.9964423942006735)

IBS: 0.229


In [71]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [72]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [73]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 22:15:10,945] A new study created in memory with name: no-name-1181e616-5fb6-48c6-b17a-30d116f64d61


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-14 22:15:12,355] Trial 0 finished with value: 0.6249682148055231 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6566523605150214
[I 2024-04-14 22:15:19,669] Trial 1 finished with value: 0.6241098457068107 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6781115879828327
[I 2024-04-14 22:16:27,927] Trial 19 finished with value: 0.6415416799528819 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 11 with value: 0.6491468763533488.
Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6615969581749049
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 22:16:36,107] Trial 20 finished with value: 0.6383398348532772 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 11 with value: 0.6491468763533488.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5617021276595745

Fold 5 C-index: 0.6781115879828327
[I 2024-04-14 22:18:11,639] Trial 37 finished with value: 0.6399802791763081 and parameters: {'subsample': 0.15245954770280346, 'dropout_rate': 0.9117253364549719, 'n_estimators': 230, 'learning_rate': 0.05621431261796065}. Best is trial 28 with value: 0.6503096338516874.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 22:18:19,932] Trial 38 finished with value: 0.6267465093526564 and parameters: {'subsample': 0.5602770145716328, 'dropout_rate': 0.5364701773999812, 'n_estimators': 404, 'learning_rate': 0.07843779087354608}. Best is trial 28 with value: 0.6503096338516874.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 22:18:21,465] Trial 39 finished with value: 0.6305782657

...

Fold 3 IBS: 0.2756019255493613
Fold 4 IBS: 0.2426706185564074
Fold 5 IBS: 0.22283081379821795
[I 2024-04-14 22:24:08,148] Trial 19 finished with value: 0.24310221296462103 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.24550608887605468
Fold 2 IBS: 0.2133505584021398
Fold 3 IBS: 0.23294747738469943
Fold 4 IBS: 0.23024731593258208
Fold 5 IBS: 0.2123372739360029
[I 2024-04-14 22:24:08,573] Trial 20 finished with value: 0.22687774290629575 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2448259892509062
Fold 2 IBS: 0.21786672573833757
Fold 3 IBS: 0.23050391968540718
Fold 4 IBS: 0.23249100529840805
Fold 5 IBS: 0.215533050087791
[I 2024-04-14 22:24:09,005] Trial 21 finis

Fold 3 IBS: 0.2566141669263308
Fold 4 IBS: 0.23453614765450895
Fold 5 IBS: 0.2122914778844625
[I 2024-04-14 22:24:28,671] Trial 38 finished with value: 0.2333975796328732 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2943463281154152
Fold 2 IBS: 0.21435275898375683
Fold 3 IBS: 0.292776623523439
Fold 4 IBS: 0.2606234117164125
Fold 5 IBS: 0.24191548716779984
[I 2024-04-14 22:24:29,042] Trial 39 finished with value: 0.26080292190136467 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2956705905294083
Fold 2 IBS: 0.21430445848659016
Fold 3 IBS: 0.2928588854990794
Fold 4 IBS: 0.2596622123064148
Fold 5 IBS: 0.24678618384472736
[I 2024-04-14 22:24:29,731] Trial 40 finished 

Fold 4 IBS: 0.28596472610806994
Fold 5 IBS: 0.27461860801929316
[I 2024-04-14 22:24:40,225] Trial 57 finished with value: 0.29238736666151666 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2524958013314946
Fold 2 IBS: 0.20467957580837548
Fold 3 IBS: 0.24466604990129345
Fold 4 IBS: 0.22951388023914565
Fold 5 IBS: 0.21232254648299242
[I 2024-04-14 22:24:40,579] Trial 58 finished with value: 0.22873557075266032 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2450766728615442
Fold 2 IBS: 0.21570571438138603
Fold 3 IBS: 0.23170031253784298
Fold 4 IBS: 0.23100841297079955
Fold 5 IBS: 0.21387190006582926
[I 2024-04-14 22:24:40,809] Trial 59 finished with value: 0.227472602563

Fold 5 IBS: 0.21495710360760373
[I 2024-04-14 22:25:07,504] Trial 76 finished with value: 0.22786680538358572 and parameters: {'subsample': 0.5683268745859826, 'dropout_rate': 0.8653081987830238, 'n_estimators': 34, 'learning_rate': 0.020347121996093395}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2506811440240583
Fold 2 IBS: 0.2053539697592853
Fold 3 IBS: 0.2432701864482772
Fold 4 IBS: 0.2284683602008423
Fold 5 IBS: 0.21006566848816424
[I 2024-04-14 22:25:08,691] Trial 77 finished with value: 0.22756786578412544 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.24728609182637104
Fold 2 IBS: 0.20948456237788454
Fold 3 IBS: 0.2367879689804507
Fold 4 IBS: 0.22900068533758589
Fold 5 IBS: 0.21042389725248523
[I 2024-04-14 22:25:09,301] Trial 78 finished with value: 0.2265966411549555 and parameters: {'subsampl

Fold 1 IBS: 0.3528807852002724
Fold 2 IBS: 0.26163876419301324
Fold 3 IBS: 0.35927296600690356
Fold 4 IBS: 0.2978922662575483
Fold 5 IBS: 0.3041795583962892
[I 2024-04-14 22:25:24,483] Trial 96 finished with value: 0.31517286801080535 and parameters: {'subsample': 0.41313939490511475, 'dropout_rate': 0.5213109482382285, 'n_estimators': 158, 'learning_rate': 0.054487916689644846}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.24708172105946397
Fold 2 IBS: 0.21005062605084635
Fold 3 IBS: 0.23658239945020892
Fold 4 IBS: 0.2291089640988238
Fold 5 IBS: 0.2105403389046395
[I 2024-04-14 22:25:25,466] Trial 97 finished with value: 0.2266728099127965 and parameters: {'subsample': 0.5389995281019297, 'dropout_rate': 0.6077476353900257, 'n_estimators': 73, 'learning_rate': 0.014931416501109941}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.26183286577938214
Fold 2 IBS: 0.20183342233582774
Fold 3 IBS: 0.2575253907596287
Fold 4 IBS: 0.23417362585495322
Fold 5 IBS:

In [74]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [75]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.65
train_ibs:  0.227


#### Test

In [76]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [77]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8178252892822001,
                                              learning_rate=0.014116661098438898,
                                              n_estimators=383,
                                              random_state=123,
                                              subsample=0.10146809971965022)

C-index score: 0.531


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6341528845468055,
                                              learning_rate=0.014291926646619518,
                                              n_estimators=76, random_state=123,
                                              subsample=0.6070545538128025)

IBS: 0.234


In [78]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [79]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.845,1.0
ExtraSurvivalTrees,0.837,2.0
GradientBoosting,0.711,3.0
CoxElastic,0.705,4.0
CoxPH,0.703,5.0
CoxLasso,0.701,6.0
ComponentwiseGradientBoosting,0.650,7.0
CoxRidge,0.648,8.0


In [80]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxPH,0.199,2.0
CoxLasso,0.199,2.0
CoxElastic,0.199,2.0
Randomsurvivalforest,0.205,4.0
ExtraSurvivalTrees,0.207,5.0
ComponentwiseGradientBoosting,0.227,6.0
GradientBoosting,0.234,7.0
CoxRidge,0.236,8.0


In [81]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.537,1.0
ComponentwiseGradientBoosting,0.531,2.0
CoxPH,0.528,3.5
CoxLasso,0.528,3.5
CoxElastic,0.527,5.0
Randomsurvivalforest,0.521,6.0
GradientBoosting,0.517,7.0
ExtraSurvivalTrees,0.509,8.0


In [82]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.234,3.0
Randomsurvivalforest,0.248,4.0
ExtraSurvivalTrees,0.259,5.0
CoxLasso,0.308,6.5
CoxElastic,0.308,6.5
CoxPH,0.309,8.0


In [85]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/standard/plsr/'  # Folder path where you want to save the files

# List of corresponding file namess
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_standard_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [86]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
